In [ ]:
# -*- coding: utf-8 -*-
import sys
import os
import gc
import json
import h5py
import pandas as pd
import numpy as np
from tqdm import tqdm

# ==============================================================================
# CONFIGURATION - PASTIKAN PATH BENAR
# ==============================================================================
CSV_PATH = '/Volumes/Extreme SSD/stream_stead/data_stead/merge.csv'
HDF5_PATH = '/Volumes/Extreme SSD/stream_stead/data_stead/merge.hdf5'
# Path JSON Zhi Geng untuk anti-leakage
ZHI_GENG_JSON = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mcquake_ori_file/Benchmark_ STEAD 3C_ test n15275 r100/STEAD data, test n15275 r100.json'

# Output data matang (Benchmark 100k)
PREPROCESSED_H5 = '/Volumes/Extreme SSD/stream_stead/data_stead/stead_processed_100k_core.h5'

def main():
    # Pastikan direktori output ada
    os.makedirs(os.path.dirname(PREPROCESSED_H5), exist_ok=True)

    print("[INFO] Memulai pipeline pengolahan data...", flush=True)

    # --------------------------------------------------------------------------
    # FASE 1: FILTERING & BALANCING (Anti-Leakage Strategy)
    # --------------------------------------------------------------------------
    print("[INFO] Memuat metadata dan melakukan anti-leakage filtering...", flush=True)
    
    with open(ZHI_GENG_JSON, 'r') as f:
        zhi_geng_data = json.load(f)
    zhi_geng_traces = set(zhi_geng_data.keys())
        
    df_raw = pd.read_csv(CSV_PATH, low_memory=False)
    
    # Hanya ambil Earthquake dan Noise yang belum pernah dipakai Zhi Geng
    df_filtered = df_raw[df_raw['trace_category'].isin(['earthquake_local', 'noise'])]
    df_unseen = df_filtered[~df_filtered['trace_name'].isin(zhi_geng_traces)]
    
    # Balance: 50.000 Earthquake + 50.000 Noise
    n_samples = 50000 
    df_eq = df_unseen[df_unseen['trace_category'] == 'earthquake_local'].sample(n=n_samples, random_state=42)
    df_noise = df_unseen[df_unseen['trace_category'] == 'noise'].sample(n=n_samples, random_state=42)
    
    df_final = pd.concat([df_eq, df_noise]).sample(frac=1, random_state=42).reset_index(drop=True)
    
    trace_names = df_final['trace_name'].to_numpy()
    trace_categories = df_final['trace_category'].to_numpy()
    p_arrivals = df_final['p_arrival_sample'].fillna(0).to_numpy().astype(np.int32)
    
    # Bersihkan memori
    del df_raw, df_filtered, df_unseen, df_eq, df_noise, df_final
    gc.collect()

    # --------------------------------------------------------------------------
    # FASE 2: PROCESSING & FREEZING
    # --------------------------------------------------------------------------
    num_points = 700   # Jendela input model: 7 detik
    norm_points = 900  # Jendela normalisasi: 9 detik
    total_target = len(trace_names)

    tqdm.write(f"[INFO] Memulai freezing ke {PREPROCESSED_H5} ({total_target} samples)...")
    
    with h5py.File(PREPROCESSED_H5, 'w') as f_out:
        dset_1c = f_out.create_dataset("X_1C", shape=(total_target, num_points, 1), dtype=np.float32, compression="gzip", compression_opts=4)
        dset_3c = f_out.create_dataset("X_3C", shape=(total_target, num_points, 3), dtype=np.float32, compression="gzip", compression_opts=4)
        dset_y  = f_out.create_dataset("Y", shape=(total_target,), dtype=np.int32)
        dset_names = f_out.create_dataset("trace_name", shape=(total_target,), dtype=h5py.string_dtype(encoding='utf-8'))

        with h5py.File(HDF5_PATH, 'r') as f_in:
            data_group = f_in['data']
            
            for idx in tqdm(range(total_target), desc="Processing Traces"):
                trace_id = trace_names[idx]
                if trace_id not in data_group: continue
                    
                raw_wave_3c = data_group[trace_id][()].astype(np.float32)
                raw_wave_3c -= np.mean(raw_wave_3c, axis=0) # Detrending
                
                # Slicing & Normalisasi Independen per Saluran
                if trace_categories[idx] == 'earthquake_local':
                    start = int(p_arrivals[idx])
                    if start < 0 or (start + norm_points) > len(raw_wave_3c): continue
                    
                    wave_3c = raw_wave_3c[start:start+num_points, :]
                    norm_val = np.max(np.abs(raw_wave_3c[start:start+norm_points, :]), axis=0)
                    label = 1
                else:
                    wave_3c = raw_wave_3c[:num_points, :]
                    norm_val = np.max(np.abs(raw_wave_3c[:norm_points, :]), axis=0)
                    label = 0
                
                norm_val[norm_val == 0] = 1e-8
                wave_3c /= norm_val
                
                # Write to HDF5
                dset_3c[idx] = wave_3c
                dset_1c[idx] = wave_3c[:, 2].reshape(num_points, 1) # Z component
                dset_y[idx] = label
                dset_names[idx] = trace_id

    tqdm.write("\n=======================================================")
    tqdm.write(" [SUKSES] BERKAS BINER PRE-DUMP STEAD 100K BERHASIL DICETAK!")
    tqdm.write("=======================================================")

if __name__ == "__main__":
    main()

[INFO] Memulai pipeline pengolahan data...
[INFO] Memuat metadata dan melakukan anti-leakage filtering...
[INFO] Memulai freezing ke /Volumes/Extreme SSD/stream_stead/data_stead/stead_processed_100k_core.h5 (100000 samples)...


Processing Traces:   1%|          | 599/100000 [01:02<4:23:43,  6.28it/s]